<a href="https://colab.research.google.com/github/almagomen/v2v/blob/opencut_launch/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video + OpenCut – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `FISH_API_KEY`
3. **Modelos en Drive**: Si tu carpeta `v2v` está en **Compartido conmigo** en Drive, hazle clic derecho ➔ *Organizar* ➔ *Añadir acceso directo* ➔ elige **Mi unidad**.
4. Ejecuta la celda para iniciar FastAPI y el editor OpenCut.

In [ ]:
import os
import sys
import warnings
from pathlib import Path
from google.colab import drive, userdata, output
warnings.filterwarnings("ignore", category=UserWarning)

# 1. Montaje de Google Drive
if 'initial_setup' not in globals():
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if not os.path.exists("/content/drive/MyDrive/v2v"):
        print("💡 Recordatorio: Si tu carpeta 'v2v' está en 'Compartido conmigo', añade un acceso directo a 'Mi unidad' en Google Drive para que Colab acceda a tus modelos.")

# 2. Exportar Secrets al Entorno para subprocesos (FastAPI/Uvicorn)
for key in ["GOOGLE_API_KEY", "FISH_API_KEY"]:
    try:
        val = userdata.get(key)
        if val: os.environ[key] = val
    except Exception: pass
os.environ["DRIVE_MOUNT"] = "true"
os.environ["V2V_BASE_DIR"] = "/content"

# 3. Gestión del Repositorio (Clone o Pull en rama opencut_launch)
repo_path = "/content/v2v"
branch_name = "opencut_launch"
get_ghp = userdata.get('GHP_GITHUB_TOKEN')
repo_url = f"https://{get_ghp}@github.com/almagomen/v2v.git"

if not os.path.exists(repo_path):
    !git clone -q -b {branch_name} {repo_url} {repo_path}
    %cd {repo_path}
else:
    %cd {repo_path}
    !git checkout {branch_name}
    !git pull

# 4. Instalación de Dependencias (NumPy 1.x requerido por PyTorch/Pyannote/Whisper)
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias de Python y FastAPI...")
    !uv pip install "numpy<2" -q
    !uv pip install -r requirements.txt -q
    !uv pip install "yt-dlp[default]" bs-roformer-infer -q
    !uv pip install -U git+https://github.com/jianfch/stable-ts.git --no-deps -q
    !uv pip install openai-whisper --no-deps -q
    !uv pip install git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' sentencex 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' 'torch-audiomentations>=0.12.0' torchcodec==0.7.0 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    deps_installed = True

# 5. Instalación de Bun para el frontend OpenCut
if 'bun_installed' not in globals():
    print("⚡ Instalando Bun para OpenCut...")
    !curl -fsSL https://bun.sh/install | bash -s "bun-v1.2.18"
    os.environ["PATH"] = f"/root/.bun/bin:{os.environ['PATH']}"
    %cd /content/v2v/opencut/apps/web
    !bun install --no-save
    %cd /content/v2v
    bun_installed = True

# 6. Precarga y verificación de Modelos IA
if 'models_loaded' not in globals():
    print("🛠️ Precargando modelos IA (Whisper, Pyannote, RoFormer)...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    get_whisper_model()
    get_diarization_model()
    try:
        from src.download import get_roformer_model
        get_roformer_model()
    except Exception as e:
        print(f"⚠️ Aviso sobre BS-RoFormer: {e}")
    models_loaded = True

initial_setup = True

# 🌐 7. Túnel nativo de Google Colab para abrir OpenCut en ventana nueva
output.serve_kernel_port_as_window(5173, anchor_text="🚀 Abrir Editor OpenCut + V2V")

# 8. Iniciar servidores
!python launch_opencut.py
